# AdaFace 100k - Colab Restore, Benchmark, and Smoke Test
This notebook mounts Google Drive, copies the 100k dataset TAR archives to fast local `/content` storage, runs validation, executes batch size benchmarks, and performs diagnostic smoke tests.

In [ ]:
# PHASE 1: Mount Google Drive & Environment Checks
import os
import shutil
import hashlib
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

drive_root = Path("/content/drive/MyDrive/adaFace-noise-study")
splits_tar = drive_root / "splits.tar"
corrupted_tar = drive_root / "corrupted.tar"
train_rec = drive_root / "raw" / "faces_webface_112x112" / "train.rec"
train_idx = drive_root / "raw" / "faces_webface_112x112" / "train.idx"

print("==================================================")
print("PHASE 1 - DRIVE VERIFICATION")
print("==================================================")
for file_path in [splits_tar, corrupted_tar, train_rec, train_idx]:
    if file_path.exists():
        size_gb = file_path.stat().st_size / (1024**3)
        print(f"[FOUND] {file_path.name} - Size: {size_gb:.2f} GB")
    else:
        raise RuntimeError(f"[MISSING] {file_path} does not exist on Drive!")

print("\nChecking disk space...")
total, used, free = shutil.disk_usage("/content")
print(f"Free space in /content: {free / (1024**3):.2f} GB")
total_d, used_d, free_d = shutil.disk_usage("/content/drive/MyDrive")
print(f"Free space in Drive: {free_d / (1024**3):.2f} GB")

print("\nCalculating checksums... (this might take a minute)")
def get_sha256(filepath):
    sha256 = hashlib.sha256()
    with open(filepath, 'rb') as f:
        for block in iter(lambda: f.read(4096 * 1024), b''):
            sha256.update(block)
    return sha256.hexdigest()

print(f"splits.tar SHA256: {get_sha256(splits_tar)}")
print(f"corrupted.tar SHA256: {get_sha256(corrupted_tar)}")


In [ ]:
%%bash
# PHASE 2: Clone GitHub Repository
echo "=================================================="
echo "PHASE 2 - CLONE REPOSITORY"
echo "=================================================="
cd /content
rm -rf adaFace-noise-study
git clone https://github.com/Raja2027/adaFace-noise-study.git
cd adaFace-noise-study
git checkout b56017ecf3765e3005b37c358bc660552884283a
git submodule update --init

echo ""
echo "DATASET BASELINE COMMIT: bcb9ae487269decfc377140dd32e75f1172cd847"
echo -n "IMPLEMENTATION COMMIT: "
git rev-parse HEAD


In [ ]:
%%bash
# PHASE 4: Verify TAR contents before extraction
echo "=================================================="
echo "PHASE 4 - VERIFY TAR CONTENTS"
echo "=================================================="
echo "Inspecting splits.tar..."
tar -tf /content/drive/MyDrive/adaFace-noise-study/splits.tar | grep -E "master_100k.csv|clean_high_train.csv|noise_map_20pct.csv|0pct/|20pct/" | head -n 15
echo "\nInspecting corrupted.tar..."
tar -tf /content/drive/MyDrive/adaFace-noise-study/corrupted.tar | grep "100k_blur" | head -n 10


In [ ]:
%%bash
# PHASE 3 & 5: Restore Data and Verify
echo "=================================================="
echo "PHASE 3 & 5 - RESTORE DATA TO /content"
echo "=================================================="
cd /content/adaFace-noise-study
mkdir -p data

echo "Extracting splits.tar..."
tar -xf /content/drive/MyDrive/adaFace-noise-study/splits.tar -C data/

echo "Extracting corrupted.tar..."
tar -xf /content/drive/MyDrive/adaFace-noise-study/corrupted.tar -C data/

echo "\nExtraction complete. Verifying counts..."
hq_train=$(find data/splits/100k/images/train -type f -name "*.png" | wc -l)
hq_heldout=$(find data/splits/100k/images/heldout -type f -name "*.png" | wc -l)
lq_train=$(find data/corrupted/100k_blur_15x15_s5/train -type f -name "*.png" | wc -l)

echo "HQ Train images: $hq_train (Expected: 100000)"
echo "HQ Heldout images: $hq_heldout (Expected: 8000)"
echo "LQ Train images: $lq_train (Expected: 100000)"

if [ $hq_train -ne 100000 ] || [ $hq_heldout -ne 8000 ] || [ $lq_train -ne 100000 ]; then
    echo "ERROR: Extraction counts do not match expectations!"
    exit 1
fi


In [ ]:
%%bash
# PHASE 6: Run Data Validator
echo "=================================================="
echo "PHASE 6 - RUN EXISTING DATA VALIDATOR"
echo "=================================================="
cd /content/adaFace-noise-study
python scripts/validate_100k_dataset.py


In [ ]:
%%bash
# PHASE 7: Verify Probe Conditions
echo "=================================================="
echo "PHASE 7 - VERIFY/CREATE PROBE CONDITIONS"
echo "=================================================="
cd /content/adaFace-noise-study
python scripts/generate_probe_conditions.py


In [ ]:
%%bash
# PHASE 8: GPU Environment Report
echo "=================================================="
echo "PHASE 8 - GPU ENVIRONMENT REPORT"
echo "=================================================="
nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader
nvcc --version | grep "release"
python -c "import torch; print(f'PyTorch: {torch.__version__}')"
python --version
cd /content/adaFace-noise-study && echo -n "Git commit: " && git rev-parse HEAD
echo "Dataset baseline commit: bcb9ae487269decfc377140dd32e75f1172cd847"


In [ ]:
%%bash
# PHASE 9: Real Pipeline Benchmark
echo "=================================================="
echo "PHASE 9 - REAL PIPELINE BENCHMARK"
echo "=================================================="
cd /content/adaFace-noise-study
python scripts/benchmark_100k.py


In [ ]:
%%bash
# PHASE 10-13: Smoke Tests, Checkpoint, Delta_G, Diagnostic non-interference
echo "=================================================="
echo "PHASE 10-13 - SMOKE TESTS"
echo "=================================================="
cd /content/adaFace-noise-study
python scripts/smoke_test_runner.py


In [ ]:
# PHASE 14 & 15: Stop & Final Report
print("==================================================")
print("PHASE 15 - FINAL REPORT (See output above)")
print("==================================================")
print("DATA RESTORE: COMPLETE")
print("DATA VALIDATION: COMPLETE")
print("ENVIRONMENT: PRINTED")
print("BENCHMARK: COMPLETE")
print("SMOKE TESTS: COMPLETE")
print("CHECKPOINT RESTORE: COMPLETE")
print("DELTA_G: COMPLETE")
print("DIAGNOSTIC NON-INTERFERENCE: COMPLETE")
print("\nFINAL STATUS: READY FOR FULL TRAINING (Pending review of above logs)")
